# Module 3 — Enriching the Graph with Structured Data

**The gap, from Module 2:** `Company → Document → Chunk` only holds what's inside the 10-Ks.
Officers are named there, but their career history is "incorporated by reference" to the proxy
statement — a document never ingested. Filing text also has no notion of a company's
quantitative fundamentals, corporate structure, or dated corporate events — those live in
line-item tables and prose that don't retrieve reliably. No amount of retrying in the agentic
loop fixes a graph coverage gap; it needs more graph.

**What we build:** three structured data sources, each a genuinely different *kind* of source —
Wikidata (crowdsourced structured KB, queried via SPARQL: executives/board with career history,
company profile, corporate structure), Sharadar (paid financial-data vendor, queried via its own
REST API: fundamentals, corporate actions), and the NYT Article Search API (general news
search) — writing `Person`, `Article`, `FinancialPeriod`, and `Event` nodes (plus profile
properties and `SUBSIDIARY_OF` edges on `Company`) into the same graph. Three new retrieval tools
(`get_executives`, `get_company_profile`, `get_financials`) join the two filing-text tools from
Module 2, and a real bug in how the strategy prompt described tools gets fixed along the way.

**New components introduced:**
- `enrichment.loaders` — `load_executives`/`load_company_profile` (Wikidata SPARQL),
  `load_fundamentals`/`load_corporate_actions`/`load_sector_classification` (Sharadar REST),
  `load_news` (NYT Article Search API)
- `enrichment.graph_writer` — `write_executives`/`write_company_profile`/`write_financials`/
  `write_events`/`write_sector_classification`/`write_news`
- `ingestion.schema.apply_enrichment_schema()` — constraints + fulltext index for `Person`,
  `Event`, `Article`, `FinancialPeriod`
- `retrieval.graph_nav`/`agent.tools` — `get_executives`, `get_company_profile`,
  `get_financials`; `agent.tools.MODULE_3_TOOLS` is the full six-tool set
- `agent.prompts.MODULE_3_STRATEGY_PROMPT` — sequencing guidance only; tool source/usage now
  lives solely in each tool's docstring (see section 9 for why)

> Constraints are idempotent (`IF NOT EXISTS`) and every writer `MERGE`s, so all cells below are
> safe to re-run.


## 1. Recap — Where Module 2 Left Off

Module 2 closed by asking the agent a people question the document graph can't answer: *which
other companies have 3M's current executive officers previously worked at or served as directors
of?* `semantic_search`/`fulltext_search` could only surface the 10-K's own text, which names the
officers but explicitly defers their career history to 3M's proxy statement — a document outside
this corpus. We'll come back to that exact question at the end of this notebook, once the graph
actually holds the answer.


## 2. Schema — `Person`, `Event`, `Article`, `FinancialPeriod`

Same pattern as Module 1's `apply_basic_schema()`: uniqueness constraints plus a fulltext index,
defined in `ingestion.schema` as `CONSTRAINTS_M3`/`INDEXES_M3` and applied idempotently.
`Person` and `Article` get populated first (sections 3-5, below); `Event` and `FinancialPeriod`
have to wait until section 7-8 introduce Sharadar, the source that actually backs them.

In [1]:
from financial_advisor.ingestion.schema import apply_enrichment_schema

apply_enrichment_schema()


  [schema] OK  person_id
  [schema] OK  event_id
  [schema] OK  article_id
  [schema] OK  financial_period_id
  [schema] OK  article_text
[schema] 5/5 statements applied — all good


## 3. Executives and Board Members — Wikidata

`load_executives(company_id)` runs a SPARQL query against Wikidata for officers (CEO `P169`,
chairperson `P488`) and board members (`P3320`), then augments each person with a Wikipedia
summary and their own career history (`P108` employer claims) — this is the data Module 2
needed. Board members are scoped to `FILING_YEAR = 2025` (the more recent of the two filing
years already in the graph) so the roster matches the 10-Ks already ingested; the CEO/chairperson
title is cross-checked against the person's own tenure history for the same reason (Wikidata's
org-level claims only reflect the *current* holder, with no dates). Rate-limited to Wikidata's
public SPARQL endpoint, so this takes a little while per company.


In [2]:
from financial_advisor.enrichment.loaders import load_executives

executives_by_company = {}
for company_id in ["3M", "APPLE"]:
    executives = load_executives(company_id)
    executives_by_company[company_id] = executives
    print(f"{company_id}: {len(executives)} executives/board members")


3M: 11 executives/board members


APPLE: 9 executives/board members


In [3]:
import json

sample = executives_by_company["3M"][0]
print(json.dumps(executives_by_company["3M"][0], indent=2))
print(json.dumps(executives_by_company["3M"][1], indent=2))
print(json.dumps(executives_by_company["APPLE"][0], indent=2))


{
  "id": "Q94016219",
  "name": "Mike Roman",
  "titles": [
    {
      "title": "Board Member",
      "start": "2018-05-08T00:00:00Z",
      "end": null
    }
  ],
  "bio": null,
  "career_history": [
    {
      "employer": "Hughes Aircraft Company",
      "employer_qid": "Q196253",
      "title": null,
      "start": "1983-01-01T00:00:00Z",
      "end": "1988-01-01T00:00:00Z"
    },
    {
      "employer": "3M",
      "employer_qid": "Q159433",
      "title": null,
      "start": "1988-01-01T00:00:00Z",
      "end": null
    }
  ]
}
{
  "id": "Q125288170",
  "name": "Anne H. Chow",
  "titles": [
    {
      "title": "Board Member",
      "start": "2023-02-09T00:00:00Z",
      "end": null
    }
  ],
  "bio": null,
  "career_history": [
    {
      "employer": "AT&T",
      "employer_qid": "Q35476",
      "title": null,
      "start": "1990-01-01T00:00:00Z",
      "end": "2022-01-01T00:00:00Z"
    }
  ]
}
{
  "id": "Q265852",
  "name": "Tim Cook",
  "titles": [
    {
      "title": "

## 4. Writing Executives to the Graph

`write_executives` upserts one `Person` node per person (`id`, `name`, `bio`) and `ROLE_AT`
edges for every title — both at the target company and at each `career_history` employer.
Employers outside the two companies ingested in Module 1 land as lightweight stub `Company`
nodes (`stub: true`); we only know their name here, not their filings.


In [4]:
from financial_advisor.enrichment.graph_writer import write_executives

for company_id, executives in executives_by_company.items():
    write_executives(company_id, executives)


In [4]:
from financial_advisor.services.neo4j_service import neo4j_service

rows = neo4j_service.run_query(
    "MATCH (p:Person)-[r:ROLE_AT]->(c:Company) "
    "RETURN c.id AS company, c.stub AS is_stub, p.name AS person, r.title AS title "
    "ORDER BY company, person"
)
for row in rows:
    print(row)


{'company': '3M', 'is_stub': None, 'person': 'Amy Hood', 'title': 'Board Member'}
{'company': '3M', 'is_stub': None, 'person': 'Anne H. Chow', 'title': 'Board Member'}
{'company': '3M', 'is_stub': None, 'person': 'Audrey Choi', 'title': 'Board Member'}
{'company': '3M', 'is_stub': None, 'person': 'David Dillon', 'title': 'Board Member'}
{'company': '3M', 'is_stub': None, 'person': 'Gregory R. Page', 'title': 'Board Member'}
{'company': '3M', 'is_stub': None, 'person': 'Jim Fitterling', 'title': 'Board Member'}
{'company': '3M', 'is_stub': None, 'person': 'Mike Roman', 'title': 'Board Member'}
{'company': '3M', 'is_stub': None, 'person': 'Mike Roman', 'title': 'Employee'}
{'company': '3M', 'is_stub': None, 'person': 'Pedro Pizarro', 'title': 'Board Member'}
{'company': '3M', 'is_stub': None, 'person': 'Suzan Kereere', 'title': 'Board Member'}
{'company': '3M', 'is_stub': None, 'person': 'Thomas Brown', 'title': 'Board Member'}
{'company': '3M', 'is_stub': None, 'person': 'Thomas W. Swee

## 5. News — NYT Article Search

`load_news(company_id, start_date, end_date)` queries the NYT Article Search API and `write_news`
upserts `Article` nodes linked to the company via `MENTIONED_IN`. One honest caveat: this is a
*mentions* relationship from a general-purpose news search, not an "this article is about the
company" classifier — a company as large as 3M or Apple shows up in plenty of daily
market-roundup articles that only name it once, in passing, alongside dozens of others. That's a
different precision/recall trade-off than the Wikidata lookup above, and worth keeping in mind
when a retrieval tool eventually searches `Article.text` too.


In [2]:
from financial_advisor.enrichment.graph_writer import write_news
from financial_advisor.enrichment.loaders import load_news

for company_id in ["3M", "APPLE"]:
    articles = load_news(company_id, start_date="2024-01-01", end_date="2025-01-01")
    write_news(company_id, articles)
    print(f"{company_id}: {len(articles)} articles")


3M: 16 articles
APPLE: 30 articles


In [3]:
rows = neo4j_service.run_query(
    "MATCH (c:Company)-[:MENTIONED_IN]->(a:Article) "
    "RETURN c.id AS company, a.published_at AS published_at, a.title AS title "
    "ORDER BY published_at"
)
for row in rows:
    print(row)


NameError: name 'neo4j_service' is not defined

## 6. Company Profile & Structure — Wikidata

Executives aren't the only structured fact Wikidata carries about a company. `load_company_profile(company_id)`
pulls industry, founding year, headquarters, stock exchange, and ticker symbol, plus
parent/subsidiary relationships — all via the same SPARQL endpoint as `load_executives`, just
different Wikidata properties (industry `P452`, inception `P571`, HQ `P159`, exchange/ticker
`P414`/`P249`, parent `P749`, subsidiaries `P355`).

The `ticker` this returns matters beyond the profile itself: sections 7 and 8 use it to look up
the same company in Sharadar, a completely different kind of source (a paid financial-data
vendor feed, not a crowdsourced KB). The two connect through a fact the graph itself produced,
rather than a second hardcoded ticker table.

In [8]:
from financial_advisor.enrichment.loaders import load_company_profile

profiles_by_company = {}
for company_id in ["3M", "APPLE"]:
    profile = load_company_profile(company_id)
    profiles_by_company[company_id] = profile
    print(f"{company_id}: {profile}")

3M: {'industry': 'consumer goods industry', 'founded': 1902, 'hq': 'Maplewood', 'exchange': 'New York Stock Exchange', 'ticker': 'MMM', 'parent': None, 'subsidiaries': [{'id': 'Q21167972', 'name': '3M Innovative Properties'}, {'id': 'Q28974623', 'name': '3M (Canada)'}, {'id': 'Q28974625', 'name': '3M (Israel)'}, {'id': 'Q30256766', 'name': '3M (Germany)'}, {'id': 'Q30284639', 'name': '3M (France)'}, {'id': 'Q30289694', 'name': '3M (United Kingdom)'}]}


APPLE: {'industry': 'information technology industry', 'founded': 1976, 'hq': 'Cupertino', 'exchange': 'Nasdaq', 'ticker': 'AAPL', 'parent': None, 'subsidiaries': [{'id': 'Q421253', 'name': 'Apple Store'}, {'id': 'Q1095605', 'name': 'Claris'}, {'id': 'Q1961036', 'name': 'Beats Electronics'}, {'id': 'Q1982831', 'name': 'FileMaker, Inc.'}, {'id': 'Q2893391', 'name': 'Anobit'}, {'id': 'Q4038751', 'name': 'FingerWorks'}, {'id': 'Q7245704', 'name': 'Prismo Graphics'}, {'id': 'Q7298416', 'name': 'Raycer'}, {'id': 'Q7431115', 'name': 'SchemaSoft'}, {'id': 'Q8084438', 'name': 'Braeburn Capital'}, {'id': 'Q16835875', 'name': 'AuthenTec'}, {'id': 'Q21463549', 'name': 'Siri Inc.'}, {'id': 'Q29000247', 'name': 'Apple Germany'}, {'id': 'Q29000259', 'name': 'Apple Israel'}, {'id': 'Q105810600', 'name': 'Apple Sales International'}, {'id': 'Q111464892', 'name': 'Apple Czech'}, {'id': 'Q137672799', 'name': 'Apple Korea'}]}


`write_company_profile` sets the profile facts directly on the existing `Company` node, and
merges parent/subsidiaries as stub `Company` nodes (same `stub: true` convention as the
career-history employers in section 4) linked via `SUBSIDIARY_OF`.

In [9]:
from financial_advisor.enrichment.graph_writer import write_company_profile

for company_id, profile in profiles_by_company.items():
    write_company_profile(company_id, profile)

rows = neo4j_service.run_query(
    "MATCH (c:Company) WHERE c.ticker IS NOT NULL "
    "RETURN c.id AS company, c.industry AS industry, c.hq AS hq, "
    "c.exchange AS exchange, c.ticker AS ticker"
)
for row in rows:
    print(row)

sub_rows = neo4j_service.run_query(
    "MATCH (s:Company)-[:SUBSIDIARY_OF]->(p:Company) RETURN s.name AS subsidiary, p.name AS parent"
)
for row in sub_rows:
    print(row)

{'company': 'APPLE', 'industry': 'information technology industry', 'hq': 'Cupertino', 'exchange': 'Nasdaq', 'ticker': 'AAPL'}
{'company': '3M', 'industry': 'consumer goods industry', 'hq': 'Maplewood', 'exchange': 'New York Stock Exchange', 'ticker': 'MMM'}
{'subsidiary': '3M Innovative Properties', 'parent': None}
{'subsidiary': '3M (Canada)', 'parent': None}
{'subsidiary': '3M (Israel)', 'parent': None}
{'subsidiary': '3M (Germany)', 'parent': None}
{'subsidiary': '3M (France)', 'parent': None}
{'subsidiary': '3M (United Kingdom)', 'parent': None}
{'subsidiary': 'Apple Store', 'parent': None}
{'subsidiary': 'Claris', 'parent': None}
{'subsidiary': 'Beats Electronics', 'parent': None}
{'subsidiary': 'FileMaker, Inc.', 'parent': None}
{'subsidiary': 'Anobit', 'parent': None}
{'subsidiary': 'FingerWorks', 'parent': None}
{'subsidiary': 'Prismo Graphics', 'parent': None}
{'subsidiary': 'Raycer', 'parent': None}
{'subsidiary': 'SchemaSoft', 'parent': None}
{'subsidiary': 'Braeburn Capita

## 7. Financial Fundamentals — Sharadar

Everything so far — Wikidata, NYT — has been free/crowdsourced or a general web API. Sharadar is
a different kind of source: a **paid financial-data vendor** feed of as-reported fundamentals,
requested over its own REST API (`api.sharadar.com`, endpoint `fundamentals`) rather than SPARQL
— `enrichment.loaders._sharadar_get` is the one new piece of plumbing this needs, no new
dependency (`httpx`, same as the NYT loader). `load_fundamentals(ticker)` returns one row per
fiscal year — revenue, net income, assets, liabilities, equity, EPS — using the `ticker` section
6 pulled from Wikidata.

Treat these numbers as ground truth: they're vendor-verified figures, not an LLM's best guess at
what a 10-K's financial-statement tables say. That distinction is exactly what makes them useful
as an evaluation baseline later (Module 7): a way to check whether the agent's *own* stated
figures are factually correct, not just well-grounded in whatever text it retrieved.

While we're pulling from Sharadar, `load_sector_classification(ticker)` grabs its own
sector/industry classification too (the `tickers` endpoint) — deliberately written to
`sharadar_sector`/`sharadar_industry` rather than overwriting Wikidata's `industry` from section
6. A financial-data vendor and a crowdsourced knowledge base don't always agree on how to
classify a company, and that disagreement is worth seeing rather than papering over with a
silent overwrite.

In [5]:
import time

from financial_advisor.enrichment.loaders import load_fundamentals, load_sector_classification

fundamentals_by_company = {}
classifications_by_company = {}
for i, (company_id, profile) in enumerate(profiles_by_company.items()):
    if i > 0:
        print(f"Sleeping for 2 seconds before loading fundamentals for {company_id} ({profile['ticker']})...")
        time.sleep(2)  # light courtesy pacing against a paid API, not a documented requirement
    ticker = profile["ticker"]
    periods = load_fundamentals(ticker)
    print(f"Sleeping for 2 seconds before loading sector classification for {company_id} ({ticker})...")
    time.sleep(2)  # light courtesy pacing against a paid API, not a documented requirement
    classification = load_sector_classification(ticker)
    fundamentals_by_company[company_id] = (ticker, periods)
    classifications_by_company[company_id] = classification
    print(f"{company_id} ({ticker}): sharadar classification = {classification}")
    print(f"{company_id} ({ticker}): {len(periods)} fiscal year(s)")
    for p in periods:
        print(f"  {p}")

NameError: name 'profiles_by_company' is not defined

In [11]:
from financial_advisor.enrichment.graph_writer import write_financials, write_sector_classification

for company_id, (ticker, periods) in fundamentals_by_company.items():
    write_financials(company_id, ticker, periods)
    write_sector_classification(company_id, classifications_by_company[company_id])

rows = neo4j_service.run_query(
    "MATCH (c:Company)-[:HAS_FINANCIALS]->(fp:FinancialPeriod) "
    "RETURN c.id AS company, fp.calendardate AS year, fp.revenue AS revenue, fp.netinc AS netinc "
    "ORDER BY company, year"
)
for row in rows:
    print(row)

rows = neo4j_service.run_query(
    "MATCH (c:Company) WHERE c.sharadar_sector IS NOT NULL "
    "RETURN c.id AS company, c.industry AS wikidata_industry, "
    "c.sharadar_sector AS sharadar_sector, c.sharadar_industry AS sharadar_industry"
)
for row in rows:
    print(row)

{'company': '3M', 'year': '2016-12-31', 'revenue': '30109000000', 'netinc': '5050000000'}
{'company': '3M', 'year': '2017-12-31', 'revenue': '31657000000', 'netinc': '4858000000'}
{'company': '3M', 'year': '2018-12-31', 'revenue': '32765000000', 'netinc': '5349000000'}
{'company': '3M', 'year': '2019-12-31', 'revenue': '32136000000', 'netinc': '4570000000'}
{'company': '3M', 'year': '2020-12-31', 'revenue': '32184000000', 'netinc': '5384000000'}
{'company': '3M', 'year': '2021-12-31', 'revenue': '35355000000', 'netinc': '5921000000'}
{'company': '3M', 'year': '2022-12-31', 'revenue': '34229000000', 'netinc': '5777000000'}
{'company': '3M', 'year': '2023-12-31', 'revenue': '32681000000', 'netinc': '-6995000000'}
{'company': '3M', 'year': '2024-12-31', 'revenue': '24575000000', 'netinc': '4173000000'}
{'company': '3M', 'year': '2025-12-31', 'revenue': '24948000000', 'netinc': '3250000000'}
{'company': 'APPLE', 'year': '2016-12-31', 'revenue': '215639000000', 'netinc': '45687000000'}
{'co

## 8. Corporate Events — Sharadar Actions

Back in section 2, `Event` got a uniqueness constraint described as "a placeholder for a richer
corporate-events model" — nothing wrote one. `load_corporate_actions(ticker)` (Sharadar's
`actions` endpoint) is what actually populates it: dated structural events like spin-offs,
mergers, splits, and name changes. Wikidata does have an equivalent-sounding property
("significant event"), but its coverage of this kind of corporate fact is thin and inconsistent
— Sharadar's purpose-built, vendor-maintained table is the more reliable source for something
we actually want to see populated.

One filtering decision worth being explicit about: routine `dividend` entries are excluded
(`EXCLUDED_ACTIONS` in `enrichment.loaders`). Checked against a live response for MMM, the
`action` taxonomy is small — `dividend`, `spinoff`, `spinoffdividend` — but dividends recur
quarterly forever (39 of 42 rows for MMM alone) and would drown out the one-off structural
events `Event` is meant to surface, like 3M's 2024 Solventum spin-off. Nothing else observed so
far warrants filtering; look at what actually comes back for these two tickers below.

In [12]:
from financial_advisor.enrichment.loaders import load_corporate_actions

events_by_company = {}
for i, (company_id, (ticker, _periods)) in enumerate(fundamentals_by_company.items()):
    if i > 0:
        print(f"Sleeping for 2 seconds before loading corporate actions for {company_id} ({ticker})...")
        time.sleep(2)  # light courtesy pacing against a paid API, not a documented requirement
    actions = load_corporate_actions(ticker)
    events_by_company[company_id] = (ticker, actions)
    print(f"{company_id} ({ticker}): {len(actions)} corporate action(s)")
    for a in actions:
        print(f"  {a}")

3M (MMM): 2 corporate action(s)
  {'date': '2024-04-01', 'action': 'spinoffdividend', 'name': '3M CO', 'contraname': 'SOLVENTUM CORP'}
  {'date': '2024-04-01', 'action': 'spinoff', 'name': '3M CO', 'contraname': 'SOLVENTUM CORP'}
Sleeping for 2 seconds before loading corporate actions for APPLE (AAPL)...


APPLE (AAPL): 1 corporate action(s)
  {'date': '2020-08-31', 'action': 'split', 'name': 'APPLE INC', 'contraname': 'N/A'}


In [13]:
from financial_advisor.enrichment.graph_writer import write_events

for company_id, (ticker, actions) in events_by_company.items():
    write_events(company_id, ticker, actions)

rows = neo4j_service.run_query(
    "MATCH (c:Company)-[:HAD_EVENT]->(e:Event) "
    "RETURN c.id AS company, e.date AS date, e.type AS type, e.description AS description "
    "ORDER BY company, date"
)
for row in rows:
    print(row)

{'company': '3M', 'date': '2024-04-01', 'type': 'spinoffdividend', 'description': 'SOLVENTUM CORP'}
{'company': '3M', 'date': '2024-04-01', 'type': 'spinoff', 'description': 'SOLVENTUM CORP'}
{'company': 'APPLE', 'date': '2020-08-31', 'type': 'split', 'description': 'APPLE INC'}


## 9. Three NEW Retrieval Tools

`retrieval.graph_nav.get_executives` — a structured Cypher lookup, not a vector/fulltext search
— was already covered when Module 3 first shipped: for a company, it returns each `Person`'s
roles there plus their `career_history` at every *other* company they have a `ROLE_AT` edge to.

This session adds two more of the same shape: `get_company_profile` (industry/structure from
section 6, Sharadar's classification and the `Event` data from sections 7-8) and
`get_financials` (the `FinancialPeriod` series from section 7). All three, plus the two
filing-text tools, make up `agent.tools.MODULE_3_TOOLS` — the set the agent is built with below.

In [14]:
from financial_advisor.agent.tools import get_company_profile, get_executives, get_financials

print(get_company_profile.__doc__)

Tool that can operate on any number of inputs.


In [15]:
hits = get_executives.invoke({"company_id": "3M"})
print(f"get_executives('3M') -> {len(hits)} people\n")
for h in hits:
    roles = [r["title"] for r in h["roles"]]
    print(f"  {h['name']}  roles={roles}")
    if h["career_history"]:
        history = [(c["company"], c["title"]) for c in h["career_history"]]
        print(f"    career_history: {history}\n")

profile = get_company_profile.invoke({"company_id": "3M"})
print(f"\nget_company_profile('3M') -> {profile}")

financials = get_financials.invoke({"company_id": "3M"})
print(f"\nget_financials('3M') -> {len(financials)} fiscal year(s)")
for f in financials:
    print(f"  {f}")

get_executives('3M') -> 11 people

  Thomas W. Sweet  roles=['Board Member']
  Pedro Pizarro  roles=['Board Member']
  Suzan Kereere  roles=['Board Member']
  Jim Fitterling  roles=['Board Member']
  David Dillon  roles=['Board Member']
  Thomas Brown  roles=['Board Member']
  Audrey Choi  roles=['Board Member']
    career_history: [('Morgan Stanley', 'Employee')]

  Amy Hood  roles=['Board Member']
    career_history: [('Microsoft', 'Employee'), ('Goldman Sachs', 'Employee')]

  Anne H. Chow  roles=['Board Member']
    career_history: [('AT&T', 'Employee')]

  Mike Roman  roles=['Board Member', 'Employee']
    career_history: [('Hughes Aircraft Company', 'Employee')]

  Gregory R. Page  roles=['Board Member']
    career_history: [('Cargill', 'Employee')]


get_company_profile('3M') -> {'id': '3M', 'name': None, 'industry': 'consumer goods industry', 'founded': 1902, 'hq': 'Maplewood', 'exchange': 'New York Stock Exchange', 'ticker': 'MMM', 'sharadar_sector': 'Industrials', 'sharadar_i

## 10. Closing the Loop — Six Questions, Six New Capabilities

`build_agent` now takes an explicit `(tools, strategy_prompt)` pair — here `MODULE_3_TOOLS` and
`MODULE_3_STRATEGY_PROMPT`, matching Module 2's `build_agent(MODULE_2_TOOLS,
MODULE_2_STRATEGY_PROMPT)` call. Same retrieval loop as Module 2; the only things that changed
are what's in the graph and which tools the strategy agent can reach for.

Six questions below. The first five were unanswerable, or only vaguely answerable from filing
text, before this session's additions — each is paired with the specific capability it
exercises. The sixth is the original Module 2 struggle-question, kept as a check that nothing
regressed.

In [6]:
from financial_advisor.agent.graph import build_agent
from financial_advisor.agent.prompts import MODULE_3_STRATEGY_PROMPT
from financial_advisor.agent.state import initial_state
from financial_advisor.agent.tools import MODULE_3_TOOLS

agent = build_agent(MODULE_3_TOOLS, MODULE_3_STRATEGY_PROMPT)


def ask(question: str) -> dict:
    result = agent.invoke(initial_state(question), {"recursion_limit": 50})
    print(
        f"\n[{result['retrieval_iterations']} retrieval round(s), "
        f"{result['answer_attempts']} answer attempt(s), "
        f"{len(result['retrieved_chunks'])} item(s) retrieved]"
    )
    print(f"\nA: {result['answer']}")
    return result

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

### 10a. Profile — Where Wikidata and Sharadar Disagree

Neither the filing text nor either source alone frames this as a comparison.
`get_company_profile` carries both classifications side by side, so the agent can point out a
disagreement instead of silently picking one.

In [7]:
_ = ask(
    "What sector and industry does Wikidata classify 3M under, and does Sharadar's own "
    "classification agree?"
)

[strategy] iteration 1: 1 tool call(s) planned
    - get_company_profile({'company_id': '3M'})
[tools] get_company_profile({'company_id': '3M'}) -> 1 chunk(s)
[grade-retrieval] sufficient=False
    feedback: Missing exactly the Wikidata classification (the sector and industry values as given by Wikidata). To answer the question precisely we need the Wikidata entry for 3M (the values of the relevant properties such as industry (P452) and any sector-related property). Next steps to obtain this: 
- Query Wikidata for the company labelled "3M" (or for ticker MMM) — either fetch the entity page for 3M via the Wikidata API or run a SPARQL query on the Wikidata Query Service to return the company's industry (P452) and any sector classification. 
- Return the Wikidata-sourced sector and industry values (with Wikidata Q-IDs or property values) so we can directly compare them to Sharadar's "Industrials / Conglomerates" classification.
[strategy] iteration 2: 1 tool call(s) planned
    - get_comp

### 10b. Corporate Structure — Subsidiaries

A pure graph traversal (`SUBSIDIARY_OF`) that filing text doesn't reliably enumerate — the 10-K
lists major subsidiaries inconsistently across companies, but the graph structure doesn't care.

In [18]:
_ = ask("What subsidiaries does 3M have, according to the graph?")

[strategy] iteration 1: 1 tool call(s) planned
    - get_company_profile({'company_id': '3M'})
[tools] get_company_profile({'company_id': '3M'}) -> 1 chunk(s)


[grade-retrieval] sufficient=True


[answer] attempt #1


[grade-answer] accepted=True next_action=end

[1 retrieval round(s), 1 answer attempt(s), 1 item(s) retrieved]

A: According to the graph, 3M's subsidiaries are:
- 3M Innovative Properties (doc_id=3M)
- 3M (Canada) (doc_id=3M)
- 3M (Israel) (doc_id=3M)
- 3M (Germany) (doc_id=3M)
- 3M (France) (doc_id=3M)
- 3M (United Kingdom) (doc_id=3M)


### 10c. Financial Fundamentals — Revenue and Net Income Compared

Module 1's baseline (and Module 2's agent) could only get at figures like this by finding and
reading each company's Consolidated Statement of Income chunk — one company, one filing year, at
a time. `get_financials` returns the whole multi-year series for each company directly, no text
parsing involved.

In [19]:
_ = ask(
    "How do 3M's and Apple's revenue and net income compare over the fiscal years the graph "
    "has data for?"
)

[strategy] iteration 1: 2 tool call(s) planned
    - get_financials({'company_id': '3M'})
    - get_financials({'company_id': 'APPLE'})
[tools] get_financials({'company_id': '3M'}) -> 10 chunk(s)
[tools] get_financials({'company_id': 'APPLE'}) -> 10 chunk(s)


[grade-retrieval] sufficient=True


[answer] attempt #1


[grade-answer] accepted=True next_action=end

[1 retrieval round(s), 1 answer attempt(s), 20 item(s) retrieved]

A: Year-by-year figures (revenue, then net income):

- 2016 — 3M: revenue 30,109,000,000 USD; net income 5,050,000,000 USD (MMM:2016-12-31). Apple: revenue 215,639,000,000 USD; net income 45,687,000,000 USD (AAPL:2016-12-31).  
- 2017 — 3M: revenue 31,657,000,000 USD; net income 4,858,000,000 USD (MMM:2017-12-31). Apple: revenue 229,234,000,000 USD; net income 48,351,000,000 USD (AAPL:2017-12-31).  
- 2018 — 3M: revenue 32,765,000,000 USD; net income 5,349,000,000 USD (MMM:2018-12-31). Apple: revenue 265,595,000,000 USD; net income 59,531,000,000 USD (AAPL:2018-12-31).  
- 2019 — 3M: revenue 32,136,000,000 USD; net income 4,570,000,000 USD (MMM:2019-12-31). Apple: revenue 260,174,000,000 USD; net income 55,256,000,000 USD (AAPL:2019-12-31).  
- 2020 — 3M: revenue 32,184,000,000 USD; net income 5,384,000,000 USD (MMM:2020-12-31). Apple: revenue 274,515,000,000 USD; net income

### 10d. Corporate Events — the Timeline

This is what section 8 actually built the `Event` node for: a dated timeline that section 2
could only promise as "a placeholder for a richer corporate-events model."

In [10]:
_ = ask("What corporate actions (splits, spinoffs, mergers, name changes) has 3M had, and when?")

[strategy] iteration 1: 1 tool call(s) planned
    - get_company_profile({'company_id': '3M'})
[tools] get_company_profile({'company_id': '3M'}) -> 1 chunk(s)
[grade-retrieval] sufficient=False
    feedback: Not sufficient. Missing items required to answer the question fully and precisely: the complete, dated history of 3M’s stock splits (dates and split ratios), all spinoffs/divestitures with dates and names, any mergers or material acquisitions involving 3M (dates and counterparties), and any corporate name changes (old name, new name, and date). Suggested next queries / documents to retrieve:

1) Query the company events/corporate_actions table for doc_id=3M filtering types: split, spinoff, spin-off, divestiture, merger, acquisition, name_change. (e.g., search: "doc_id:3M AND event_type:split OR spinoff OR merger OR name_change")
2) Fetch 3M’s investor relations / corporate history page (press releases and timeline) — it often lists name changes and major spin-offs/mergers.
3) Retri

### 10e. Cross-Source Payoff — Did a Board Change Coincide with a Corporate Event?

The actual payoff of putting all of this in *one* graph instead of answering five separate
questions from five separate tools: this needs `get_executives`' role dates *and*
`get_company_profile`'s event dates together, correlated by the agent — not something either
source, or either tool, answers alone.

In [11]:
_ = ask(
    "Did any changes to 3M's board of directors or executive officers happen around the same "
    "time as one of its corporate events, such as a spinoff?"
)

[strategy] iteration 1: 2 tool call(s) planned
    - get_company_profile({'company_id': '3M'})
    - get_executives({'company_id': '3M'})
[tools] get_company_profile({'company_id': '3M'}) -> 1 chunk(s)
[tools] get_executives({'company_id': '3M'}) -> 11 chunk(s)
[grade-retrieval] sufficient=False
    feedback: What's missing to fully and precisely answer the question:
- Any board-member appointment or departure records dated in early 2024 (especially March–May 2024) including end dates/resignations, not just start dates.
- Records of executive-officer (CEO, CFO, President, other named officers) appointments or resignations with dates in early 2024.
- Official corporate disclosures around the spinoff date: 3M press releases about the Solventum spinoff, SEC filings (Form 8-Ks) dated near 2024-04-01, and the company’s investor-relations announcements or proxy statements that list director changes and officer changes.

Recommended next queries/documents to retrieve:
1) Search 3M (MMM) SEC f

### 10f. The Original Struggle-Question — Unchanged, as a Regression Check

Same question that closed Module 2, unanswerable there. Everything above added new capability;
this confirms none of it came at the cost of the one this module was originally built to fix.

In [12]:
STRUGGLE_QUESTION = (
    "Which other companies have 3M's current executive officers previously worked at or "
    "served as directors of?"
)

result = ask(STRUGGLE_QUESTION)

[strategy] iteration 1: 1 tool call(s) planned
    - get_executives({'company_id': '3M'})
[tools] get_executives({'company_id': '3M'}) -> 11 chunk(s)
[grade-retrieval] sufficient=False
    feedback: Missing information required to answer the question precisely: (1) an authoritative list of 3M’s current executive officers (the company’s management team, e.g., CEO, CFO, other named executive officers), and (2) for each of those executive officers, documented prior employers and prior director roles. The current chunks mostly describe board members and some career histories, not the full set of 3M executive officers and their prior affiliations.

Next recommended documents/queries to retrieve: 
- 3M’s official “Leadership” or “Executive Officers” page on 3m.com (company website) — this will list current executive officers and short bios including prior employers. 
- 3M’s most recent SEC filings: the Form 10-K and the Definitive Proxy Statement (DEF 14A) — these include the list of executi

Compare this to Module 2, where the same question exhausted every retry with the agent honestly
reporting the knowledge wasn't there. Now it isn't — but notice *how* it got there:

- **Iteration 1** went straight to `get_executives('3M')` — the strategy prompt update worked,
  the agent reached for the new structured tool first instead of falling back to free-text
  search. It surfaced Mike Roman's career history (Hughes Aircraft Company) instantly and
  precisely, no LLM extraction guesswork involved.
- **`evaluate_retrieval` correctly judged that wasn't enough.** Wikidata's officer query is
  strong on the CEO and the board, but SEC disclosure rules require naming *all* executive
  officers (CFO, other named execs) — a roster Wikidata mostly doesn't carry. So the loop fell
  back to `semantic_search`/`fulltext_search` for three more rounds, which is exactly the
  Module 2 behavior, just now layered on top of a first structured hit instead of starting cold.
- Those later rounds found something worth noting: the 10-K's own Item 401 "business experience"
  disclosure does list a few years of prior employers for several named officers — a fact the
  original graph coverage gap analysis didn't anticipate. Real filings vary in how much they
  actually defer to the proxy.
- The answer accepted on the first attempt, cites `doc_id`/`chunk_id` throughout, and — this is
  the important part — explicitly names the five officers it still has no external-employer data
  for, rather than silently omitting them. That's the graph's actual current coverage, honestly
  reported.

The remaining gap (those five officers) is a real one: `get_executives` only knows what
Wikidata's community-maintained officer/board data covers, and that isn't the same as SEC Item
401(b) disclosure. Closing it would mean extracting the 10-K's own executive-officer bio table
directly — which is exactly what Module 4 does next.


## 11. What's Next

Module 4 runs LLM extraction over every chunk to pull out dynamically-typed entities and
relationships the fixed schema doesn't anticipate — including, as section 10f just showed, the
kind of executive-officer business-experience detail buried in a 10-K's own text that a
structured external source like Wikidata doesn't fully cover.

Further out, Module 7's evaluation can use this module's `FinancialPeriod` data as ground
truth — a way to check whether the agent's own stated financial figures are factually correct,
not just well-grounded in whatever text or tool output it retrieved.